In [33]:
import numpy as np

In [34]:
data = [
    [12.0, 1.5, 1, 'Wine'],
    [5.0, 2.0, 0, 'Beer'],
    [40.0, 0.0, 1, 'Whiskey'],
    [13.5, 1.2, 1, 'Wine'],
    [4.5, 1.8, 0, 'Beer'],
    [38.0, 0.1, 1, 'Whiskey'],
    [11.5, 1.7, 1, 'Wine'],
    [5.5, 2.3, 0, 'Beer']
]


In [35]:
X=np.array([row[:3] for row in data])
label_mapping = {'Wine': 0, 'Beer': 1, 'Whiskey': 2}
y=np.array([row[3] for row in data])
y=np.array([label_mapping[label] for label in y])




In [36]:
def gini_impurity(x):
  counts = np.unique(x, return_counts=True)[1]
  probabilities=counts/len(x)
  gini = 1 - np.sum(probabilities**2)
  return gini






In [37]:
def split_finder(X,y):
  best_gini=float('inf')
  feature_indices=range(X.shape[1])
  for feature_index in feature_indices:
    values = np.unique(X[:, feature_index])
    #midpoint
    thresholds = (values[:-1]+values[1:])/ 2
    for threshold in thresholds:
      #left and right seperating by threshold
      left_indices=np.where(X[:,feature_index]<=threshold)[0]
      right_indices=np.where(X[:,feature_index]>threshold)[0]
      left_gini=gini_impurity(y[left_indices])
      right_gini=gini_impurity(y[right_indices])
      #calculating total gini impurity
      total_gini=(len(left_indices)*left_gini+len(right_indices)*right_gini)/len(y)
      #finding least gini impurity for all feature,threshold pairs
      if total_gini<best_gini:
        best_gini=total_gini
        best_feature_index=feature_index
        best_threshold=threshold
      #returning best possible feature and threshold,
  return best_feature_index,best_threshold





In [38]:
class Node:
  def __init__(self,feature_index=None,threshold=None,value=None,left=None,right=None):
    self.feature_index=feature_index
    self.threshold=threshold
    self.left=left
    self.right=right
    self.value=value




In [39]:
class DecisionTree:
  def __init__(self,max_depth=4,min_samples=2):
    self.max_depth=max_depth
    self.min_samples=min_samples
    self.root=None
  def build_tree(self,X,y,depth=0):
    num_samples=len(y)
    num_unique_labels=len(np.unique(y))
    if depth>=self.max_depth or num_samples<self.min_samples or num_unique_labels==1:
      #leaf node
      labels, counts = np.unique(y, return_counts=True)
      leaf_value = labels[np.argmax(counts)]
      return Node(value=leaf_value)
    #calling function to get best feature,threshold pair
    feature,threshold=split_finder(X,y)
    #conditioning
    left_indices=np.where(X[:,feature]<=threshold)[0]
    right_indices=np.where(X[:,feature]>threshold)[0]
    #recursively building tree
    left_child=self.build_tree(X[left_indices],y[left_indices],depth+1)
    right_child=self.build_tree(X[right_indices],y[right_indices],depth+1)

    return Node(feature,threshold,None,left_child,right_child)
  def fit(self,X,y):
    self.root=self.build_tree(X,y)
  def predict(self, X):
        return np.array([self.traverse_tree(x, self.root) for x in X])
  def traverse_tree(self, x, node):
        # Base case: if node has a value, it's a leaf
        if node.value is not None:
            return node.value
        # Recursive step: check the condition
        if x[node.feature_index] <= node.threshold:
            return self.traverse_tree(x, node.left)
        return self.traverse_tree(x, node.right)


In [40]:
test_data = np.array([
    [6.0, 2.1, 0],   # Expected: Beer
    [39.0, 0.05, 1], # Expected: Whiskey
    [13.0, 1.3, 1]   # Expected: Wine
])
DT=DecisionTree()
DT.fit(X,y)
predictions=DT.predict(test_data)
class_names = np.array(["Wine","Beer","Whiskey"])
predictions = class_names[predictions]
print(predictions)

['Beer' 'Whiskey' 'Wine']
